In [ ]:
import zipfile
import json
import random

# --- CONFIGURATION ---
INPUT_ZIP_FILE = "ALS-GeoJSON.zip"
OUTPUT_TXT_FILE = "./test_data/hk_address_data_5.txt"
TOTAL_ADDRESSES_TO_GENERATE = 570
VILLAGE_ADDRESSES_TO_GENERATE = 50

# --- RANDOM DATA COMPONENTS ---
URBAN_FLOORS = [str(i) for i in range(1, 70)] + ['G', 'M'] # G/F, M/F are common
URBAN_UNITS = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'S', 'Z'] + [str(i) for i in range(1, 9999)]
SHORT_FORMS = {'FLAT': 'FLT', 'ROOM': 'RM', 'UNIT': 'UT'}
VILLAGE_HOUSE_NUMS = [str(i) for i in range(1, 200)]
VILLAGE_FLOORS = ['G', 'M', '1', '2', '3'] # G/F, M/F, 1/F, 2/F, 3/F are common for village houses


def generate_urban_sub_unit_details():
    """
    Generates plausible floor/flat/unit details for URBAN high-rises.
    Returns: (eng_full, eng_short, chi_full)
    """
    unit_type = random.choice(['FLAT', 'ROOM', 'UNIT'])
    floor = random.choice(URBAN_FLOORS)
    unit = random.choice(URBAN_UNITS)

    # English full format
    eng_full_formats = [
        f"{unit_type} {unit}, {floor}/F",
        f"{unit_type} {unit} {floor}/F"
    ]
    eng_full = random.choice(eng_full_formats)

    # English short format
    eng_short_formats = [
        f"{SHORT_FORMS[unit_type]} {unit}, {floor}/F",
        f"{SHORT_FORMS[unit_type]} {unit} {floor}/F"
    ]
    eng_short = random.choice(eng_short_formats)

    # Chinese full format
    chi_full = f"{floor}樓{unit}室"

    return eng_full, eng_short, chi_full


def generate_village_unit_details():
    """
    Generates plausible house number and floor details specifically for VILLAGE houses.
    Returns: (eng_house_floor_detail, chi_house_floor_detail, eng_floor_only, chi_floor_only)
    """
    house_num = random.choice(VILLAGE_HOUSE_NUMS)

    eng_floor_only = ""
    chi_floor_only = ""
    eng_house_prefix = "House " if random.random() < 0.8 else "" # 80% chance to include "House"

    # Add a floor (G/F, M/F, 1/F etc.) about 70% of the time
    if random.random() < 0.7:
        floor = random.choice(VILLAGE_FLOORS)
        chi_floor_only = f"{floor}樓"
        eng_floor_only = f"{floor}/F"

        # Combine house number and floor for village addresses
        eng_house_floor_detail = f"{eng_floor_only}, {eng_house_prefix}{house_num}" if eng_house_prefix else f"{eng_floor_only}, {house_num}"
        chi_house_floor_detail = f"{house_num}號{chi_floor_only}"
    else:
        # No specific floor, just house number
        eng_house_floor_detail = f"{eng_house_prefix}{house_num}"
        chi_house_floor_detail = f"{house_num}號"

    return eng_house_floor_detail, chi_house_floor_detail, eng_floor_only, chi_floor_only


def is_village_address(properties):
    """Intelligently determines if an address is a village type."""
    if not properties: return False
    try:
        addr = properties.get("Address", {}).get("PremisesAddress", {})
        chi_addr = addr.get("ChiPremisesAddress", {})
        if chi_addr:
            if chi_addr.get("ChiVillage"): return True
            if "村" in chi_addr.get("ChiStreet", {}).get("StreetName", ""): return True
        eng_addr = addr.get("EngPremisesAddress", {})
        if eng_addr:
            if eng_addr.get("EngVillage"): return True
            if "VILLAGE" in eng_addr.get("EngStreet", {}).get("StreetName", "").upper(): return True
    except (TypeError, AttributeError):
        return False
    return False


def format_chinese_address(properties, is_village):
    """Formats a Chinese address. Uses the 'is_village' flag to decide formatting strategy."""
    try:
        chi_data = properties.get("Address", {}).get("PremisesAddress", {}).get("ChiPremisesAddress", {})
        if not chi_data: return None, None, None

        region = chi_data.get("Region", "")
        district = chi_data.get("ChiDistrict", "")
        street_info = chi_data.get("ChiStreet", {})
        street_name = street_info.get("StreetName", "")
        location = street_info.get("LocationName", "")
        building_name = chi_data.get("BuildingName", "")

        line1_specific_part = "" # e.g., "10樓A室" or "123號1樓"
        line2_general_part_components = []

        if is_village:
            village_name = chi_data.get("ChiVillage", {}).get("VillageName", "")
            if "村" in street_name and not village_name:
                village_name = street_name

            village_house_floor_detail, chi_house_floor_detail, _, _ = generate_village_unit_details()
            line1_specific_part = chi_house_floor_detail
            line2_general_part_components = [region, district, location, village_name]
        else:
            # Urban address
            estate = chi_data.get("ChiEstate", {}).get("EstateName", "")
            street_no_from = street_info.get("BuildingNoFrom", "")
            street_no_to = street_info.get("BuildingNoTo", "")
            street_no = f"{street_no_from}-{street_no_to}號" if street_no_to else (f"{street_no_from}號" if street_no_from else "")

            _, _, sub_unit_chi = generate_urban_sub_unit_details()
            line1_specific_part = sub_unit_chi
            line2_general_part_components = [region, district, location, estate, street_name, street_no]

        # Combine building name with the specific part for line1
        line1_parts = [part for part in [building_name, line1_specific_part] if part]
        line1 = "".join(line1_parts).strip()

        # Combine general components for line2
        line2 = "".join([part for part in line2_general_part_components if part]).strip()

        # For Chinese, more general part (line2) usually comes before more specific (line1)
        full_address = "".join(part for part in [line2, line1] if part).strip()

        if not full_address:
            return None, None, None
        return full_address, line1, line2
    except (TypeError, AttributeError):
        return None, None, None


def format_english_address(properties, is_village):
    """Formats an English address. Uses the 'is_village' flag to decide formatting strategy."""
    # Randomly choose between comma-separated or space-separated components
    use_comma_separator = random.random() < 0.7
    separator = ", " if use_comma_separator else " "

    try:
        eng_data = properties.get("Address", {}).get("PremisesAddress", {}).get("EngPremisesAddress", {})
        if not eng_data: return None, None, None, None, None, None

        region = eng_data.get("Region", "")
        district = eng_data.get("EngDistrict", "")
        street_info = eng_data.get("EngStreet", {})
        street_name = street_info.get("StreetName", "")
        location = eng_data.get("LocationName", "")
        building_name = eng_data.get("BuildingName", "")

        line1_full_specific = ""
        line1_short_specific = ""
        line2_general_part_components = []

        if is_village:
            village_name = eng_data.get("EngVillage", {}).get("VillageName", "")
            if "VILLAGE" in street_name.upper() and not village_name:
                village_name = street_name

            eng_house_floor_detail, _, _, _ = generate_village_unit_details()
            line1_full_specific = eng_house_floor_detail
            line1_short_specific = eng_house_floor_detail # For village, short form is often the same
            line2_general_part_components = [village_name, location, district, region]
        else:
            # Urban address
            estate = eng_data.get("EngEstate", {}).get("EstateName", "")
            street_no_from = street_info.get("BuildingNoFrom", "")
            street_no_to = street_info.get("BuildingNoTo", "")
            street_no = f"{street_no_from}-{street_no_to}" if street_no_to else (street_no_from if street_no_from else "")

            sub_unit_eng_full, sub_unit_eng_short, _ = generate_urban_sub_unit_details()
            line1_full_specific = sub_unit_eng_full
            line1_short_specific = sub_unit_eng_short

            street = (street_no + " " + street_name).strip()
            line2_general_part_components = [estate, street, location, district, region]

        # Construct line1 (specific parts: unit, floor, building name)
        line1_full_parts = [part for part in [line1_full_specific, building_name] if part]
        line1_full = separator.join(line1_full_parts).strip()

        line1_short_parts = [part for part in [line1_short_specific, building_name] if part]
        line1_short = separator.join(line1_short_parts).strip()

        # Construct line2 (general parts: street, district, region)
        line2_common = separator.join([part for part in line2_general_part_components if part]).strip()

        # Construct full addresses
        full_address_full = separator.join([p for p in [line1_full, line2_common] if p]).strip()
        full_address_short = separator.join([p for p in [line1_short, line2_common] if p]).strip()

        if not full_address_full:
            return None, None, None, None, None, None
        return full_address_full, line1_full, line2_common, full_address_short, line1_short, line2_common
    except (TypeError, AttributeError):
        return None, None, None, None, None, None


def main():
    print(f"Reading all JSON files from '{INPUT_ZIP_FILE}'...")
    all_features = []

    try:
        with zipfile.ZipFile(INPUT_ZIP_FILE, 'r') as zip_ref:
            list_of_json_files = [f for f in zip_ref.namelist() if f.endswith('.geojson')]
            if not list_of_json_files:
                print("FATAL ERROR: No .geojson files found in the zip archive.")
                return
            for json_filename in list_of_json_files:
                try:
                    with zip_ref.open(json_filename) as single_json_file:
                        all_features.extend(json.load(single_json_file).get("features", []))
                except json.JSONDecodeError:
                    print(f" -> WARNING: Could not parse '{json_filename}'. Skipping it.")
    except FileNotFoundError:
        print(f"\nFATAL ERROR: The file '{INPUT_ZIP_FILE}' was not found.")
        return

    if not all_features:
        print("\nNo address features could be loaded.")
        return

    village_pool = []
    urban_pool = []
    for feature in all_features:
        if is_village_address(feature.get("properties")):
            village_pool.append(feature)
        else:
            urban_pool.append(feature)

    print(f"\nSuccessfully collected and separated {len(all_features)} addresses:")
    print(f" -> Found {len(village_pool)} village-type addresses.")
    print(f" -> Found {len(urban_pool)} urban-type addresses.")

    final_features_to_process = []
    random.shuffle(village_pool)
    num_villages = min(VILLAGE_ADDRESSES_TO_GENERATE, len(village_pool))
    final_features_to_process.extend(village_pool[:num_villages])
    random.shuffle(urban_pool)
    num_urban = min(TOTAL_ADDRESSES_TO_GENERATE - num_villages, len(urban_pool))
    final_features_to_process.extend(urban_pool[:num_urban])
    random.shuffle(final_features_to_process)
    print(f"\nSelected a final mix of {len(final_features_to_process)} addresses to generate.")

    with open(OUTPUT_TXT_FILE, mode='w', encoding='utf-8') as f:
        f.write("# [input]|[line1]|[line2]\n")
        count = 0
        for feature in final_features_to_process:
            properties = feature.get("properties")
            is_village = is_village_address(properties)

            possible_outputs = []
            chi_full, chi_l1, chi_l2 = format_chinese_address(properties, is_village)
            if chi_full:
                possible_outputs.append((chi_full, chi_l2, chi_l1)) # Chinese order: general then specific

            eng_full, eng_l1, eng_l2, eng_short, eng_l1_short, eng_l2_short = format_english_address(properties, is_village)
            if eng_full:
                possible_outputs.append((eng_full, eng_l1, eng_l2)) # English order: specific then general
            if eng_short:
                possible_outputs.append((eng_short, eng_l1_short, eng_l2_short)) # English order: specific then general

            if possible_outputs:
                chosen_full, chosen_l1, chosen_l2 = random.choice(possible_outputs)
                f.write(f"{chosen_full} | {chosen_l1} | {chosen_l2}\n")
                count += 1

    print(f"\nSuccess! Generated {count} valid formatted address lines in '{OUTPUT_TXT_FILE}'")


if __name__ == "__main__":
    main()